In [2]:
import pandas as pd
import numpy as np


In [3]:
# Load cleaned data
df = pd.read_csv("../data/clean_data.csv")

# Quick sanity check
df.head()


,country,region,gdp,gdp_per_capita,gdp_growth,population,youth_population_pct,internet_penetration,mobile_broadband,tertiary_enrollment,outbound_mobility_ratio,english_proficiency,ease_of_business,regulatory_quality,political_stability,inflation,cost_of_living,currency_stability,data_year,reliability_score
0,Australia,Asia-Pacific,1.700000e+12,65169.519110,4.253046,26018721,NaN,97.0424,109.4690,106.240761,NaN,620,1.894893,1.894893,0.983716,6.594097,77.7,60,2022,0.95
1,Brazil,Latin America,1.950000e+12,9281.332821,3.016694,210306415,NaN,80.5278,101.2450,60.390621,NaN,505,-0.220438,-0.220438,-0.395037,9.280106,33.2,60,2022,0.95
2,Canada,North America,2.190000e+12,56256.800730,4.189036,38935934,NaN,94.0000,91.2387,78.898499,NaN,590,1.677093,1.677093,0.780682,6.802801,70.2,60,2022,0.95
3,China,Asia-Pacific,1.830000e+13,12970.605640,3.134189,1412175000,NaN,75.6113,124.1950,71.579947,NaN,498,-0.418418,-0.418418,-0.451277,1.973576,41.8,90,2022,0.95
4,Germany,Europe,4.200000e+12,50506.517960,1.809258,83177813,NaN,91.6298,124.1580,77.365700,NaN,613,1.522295,1.522295,0.628492,6.872574,65.6,60,2022,0.95


In [4]:
# Identify numeric columns (exclude identifiers)
non_numeric_cols = ["country", "region"]
numeric_cols = [col for col in df.columns if col not in non_numeric_cols]

numeric_cols



['gdp',
 'gdp_per_capita',
 'gdp_growth',
 'population',
 'youth_population_pct',
 'internet_penetration',
 'mobile_broadband',
 'tertiary_enrollment',
 'outbound_mobility_ratio',
 'english_proficiency',
 'ease_of_business',
 'regulatory_quality',
 'political_stability',
 'inflation',
 'cost_of_living',
 'currency_stability',
 'data_year',
 'reliability_score']

In [5]:
# Metrics where LOWER is better (will be inverted)
inverse_metrics = [
    "inflation",
    "cost_of_living"
]

inverse_metrics


['inflation', 'cost_of_living']

In [6]:
# Create a copy for normalization
df_norm = df.copy()

for col in numeric_cols:
    if col in ["data_year", "reliability_score"]:
        continue  # skip admin columns
    
    col_min = df[col].min()
    col_max = df[col].max()
    
    # Skip columns with all missing values
    if pd.isna(col_min) or pd.isna(col_max):
        continue
    
    # Min–Max normalization
    df_norm[col + "_norm"] = (df[col] - col_min) / (col_max - col_min)
    
    # Invert if lower is better
    if col in inverse_metrics:
        df_norm[col + "_norm"] = 1 - df_norm[col + "_norm"]

df_norm.head()


,country,region,gdp,gdp_per_capita,gdp_growth,population,youth_population_pct,internet_penetration,mobile_broadband,tertiary_enrollment,...,internet_penetration_norm,mobile_broadband_norm,tertiary_enrollment_norm,english_proficiency_norm,ease_of_business_norm,regulatory_quality_norm,political_stability_norm,inflation_norm,cost_of_living_norm,currency_stability_norm
0,Australia,Asia-Pacific,1.700000e+12,65169.519110,4.253046,26018721,NaN,97.0424,109.4690,106.240761,...,1.000000,0.315516,1.000000,0.847222,0.878652,0.878652,0.784775,0.367618,0.105705,0.0
1,Brazil,Latin America,1.950000e+12,9281.332821,3.016694,210306415,NaN,80.5278,101.2450,60.390621,...,0.598599,0.226917,0.444333,0.048611,0.075198,0.075198,0.134959,0.000000,0.852349,0.0
2,Canada,North America,2.190000e+12,56256.800730,4.189036,38935934,NaN,94.0000,91.2387,78.898499,...,0.926052,0.119118,0.668633,0.638889,0.795926,0.795926,0.689083,0.339054,0.231544,0.0
3,China,Asia-Pacific,1.830000e+13,12970.605640,3.134189,1412175000,NaN,75.6113,124.1950,71.579947,...,0.479099,0.474162,0.579938,0.000000,0.000000,0.000000,0.108452,1.000000,0.708054,1.0
4,Germany,Europe,4.200000e+12,50506.517960,1.809258,83177813,NaN,91.6298,124.1580,77.365700,...,0.868442,0.473763,0.650057,0.798611,0.737130,0.737130,0.617355,0.329504,0.308725,0.0


In [7]:
# Apply reliability score penalty to all normalized metrics
for col in df_norm.columns:
    if col.endswith("_norm"):
        df_norm[col] = df_norm[col] * df["reliability_score"]

df_norm.head()


,country,region,gdp,gdp_per_capita,gdp_growth,population,youth_population_pct,internet_penetration,mobile_broadband,tertiary_enrollment,...,internet_penetration_norm,mobile_broadband_norm,tertiary_enrollment_norm,english_proficiency_norm,ease_of_business_norm,regulatory_quality_norm,political_stability_norm,inflation_norm,cost_of_living_norm,currency_stability_norm
0,Australia,Asia-Pacific,1.700000e+12,65169.519110,4.253046,26018721,NaN,97.0424,109.4690,106.240761,...,0.950000,0.299740,0.950000,0.804861,0.834719,0.834719,0.745536,0.349237,0.100419,0.00
1,Brazil,Latin America,1.950000e+12,9281.332821,3.016694,210306415,NaN,80.5278,101.2450,60.390621,...,0.568669,0.215572,0.422116,0.046181,0.071438,0.071438,0.128211,0.000000,0.809732,0.00
2,Canada,North America,2.190000e+12,56256.800730,4.189036,38935934,NaN,94.0000,91.2387,78.898499,...,0.879749,0.113162,0.635202,0.606944,0.756130,0.756130,0.654629,0.322101,0.219966,0.00
3,China,Asia-Pacific,1.830000e+13,12970.605640,3.134189,1412175000,NaN,75.6113,124.1950,71.579947,...,0.455144,0.450454,0.550942,0.000000,0.000000,0.000000,0.103030,0.950000,0.672651,0.95
4,Germany,Europe,4.200000e+12,50506.517960,1.809258,83177813,NaN,91.6298,124.1580,77.365700,...,0.825020,0.450075,0.617554,0.758681,0.700273,0.700273,0.586487,0.313029,0.293289,0.00


In [8]:
# Handle missing values in normalized features
# Strategy: fill missing with column mean (neutral, non-biased)

for col in df_norm.columns:
    if col.endswith("_norm"):
        df_norm[col] = df_norm[col].fillna(df_norm[col].mean())

df_norm.filter(like="_norm").isna().sum()


gdp_norm                     0
gdp_per_capita_norm          0
gdp_growth_norm              0
population_norm              0
internet_penetration_norm    0
mobile_broadband_norm        0
tertiary_enrollment_norm     0
english_proficiency_norm     0
ease_of_business_norm        0
regulatory_quality_norm      0
political_stability_norm     0
inflation_norm               0
cost_of_living_norm          0
currency_stability_norm      0
dtype: int64

In [12]:
# Save normalized dataset
df_norm.to_csv("../data/normalized_data.csv", index=False)

print("normalized_data.csv saved successfully")


normalized_data.csv saved successfully


In [17]:
# --- Explicit feature groups (normalized columns only) ---

economic_features = [
    "gdp_norm",
    "gdp_per_capita_norm"
]

growth_features = [
    "gdp_growth_norm"
]

readiness_features = [
    "internet_penetration_norm",
    "mobile_broadband_norm",
    "tertiary_enrollment_norm",
    "english_proficiency_norm"
]

risk_features = [
    "political_stability_norm",
    "regulatory_quality_norm",
    "ease_of_business_norm",
    "currency_stability_norm",
    "inflation_norm",
    "cost_of_living_norm"
]

print("Economic:", economic_features)
print("Growth:", growth_features)
print("Readiness:", readiness_features)
print("Risk:", risk_features)


Economic: ['gdp_norm', 'gdp_per_capita_norm']
Growth: ['gdp_growth_norm']
Readiness: ['internet_penetration_norm', 'mobile_broadband_norm', 'tertiary_enrollment_norm', 'english_proficiency_norm']
Risk: ['political_stability_norm', 'regulatory_quality_norm', 'ease_of_business_norm', 'currency_stability_norm', 'inflation_norm', 'cost_of_living_norm']


In [18]:
# Category weights (must sum to 1.0)
weights = {
    "economic": 0.25,
    "growth": 0.15,
    "readiness": 0.30,
    "risk": 0.30
}

weights


{'economic': 0.25, 'growth': 0.15, 'readiness': 0.3, 'risk': 0.3}

In [19]:
# Compute category scores (mean of features in each category)
df_norm["economic_score"] = df_norm[economic_features].mean(axis=1)
df_norm["growth_score"] = df_norm[growth_features].mean(axis=1)
df_norm["readiness_score"] = df_norm[readiness_features].mean(axis=1)
df_norm["risk_score"] = df_norm[risk_features].mean(axis=1)

df_norm[[
    "country",
    "economic_score",
    "growth_score",
    "readiness_score",
    "risk_score"
]].head()


,country,economic_score,growth_score,readiness_score,risk_score
0,Australia,0.363644,0.400268,0.751150,0.477438
1,Brazil,0.066523,0.197766,0.313134,0.180136
2,Canada,0.324748,0.389784,0.558764,0.451493
3,China,0.394730,0.217011,0.364135,0.445947
4,Germany,0.331591,0.000000,0.662832,0.432225


In [20]:
# Compute final composite score
df_norm["final_score"] = (
    weights["economic"] * df_norm["economic_score"] +
    weights["growth"] * df_norm["growth_score"] +
    weights["readiness"] * df_norm["readiness_score"] +
    weights["risk"] * df_norm["risk_score"]
)

# Rank countries (higher score = better rank)
df_norm["rank"] = df_norm["final_score"].rank(ascending=False).astype(int)

# Sort by rank
df_ranked = df_norm.sort_values("rank")

df_ranked[[
    "rank",
    "country",
    "final_score",
    "economic_score",
    "growth_score",
    "readiness_score",
    "risk_score"
]]


,rank,country,final_score,economic_score,growth_score,readiness_score,risk_score
6,1,Singapore,0.614694,0.476904,0.376511,0.920061,0.543242
0,2,Australia,0.519528,0.363644,0.400268,0.751150,0.477438
9,3,United States,0.511119,0.876325,0.115164,0.578121,0.337758
8,4,United Kingdom,0.473763,0.293730,0.547132,0.664072,0.396796
2,5,Canada,0.442732,0.324748,0.389784,0.558764,0.451493
4,6,Germany,0.411415,0.331591,0.000000,0.662832,0.432225
3,7,China,0.374258,0.394730,0.217011,0.364135,0.445947
5,8,India,0.251853,0.055472,0.950000,0.054806,0.263476
7,9,South Africa,0.218758,0.022612,0.040764,0.502051,0.187917
1,10,Brazil,0.194277,0.066523,0.197766,0.313134,0.180136


In [21]:
# Strategy guardrails (minimum acceptable thresholds)
guardrails = {
    "min_readiness": 0.40,
    "min_risk": 0.40
}

guardrails


{'min_readiness': 0.4, 'min_risk': 0.4}

In [22]:
# Apply guardrails
df_filtered = df_ranked[
    (df_ranked["readiness_score"] >= guardrails["min_readiness"]) &
    (df_ranked["risk_score"] >= guardrails["min_risk"])
].copy()

# Re-rank after filtering
df_filtered["filtered_rank"] = df_filtered["final_score"].rank(
    ascending=False
).astype(int)

df_filtered = df_filtered.sort_values("filtered_rank")

df_filtered[[
    "filtered_rank",
    "country",
    "final_score",
    "readiness_score",
    "risk_score"
]]


,filtered_rank,country,final_score,readiness_score,risk_score
6,1,Singapore,0.614694,0.920061,0.543242
0,2,Australia,0.519528,0.751150,0.477438
2,3,Canada,0.442732,0.558764,0.451493
4,4,Germany,0.411415,0.662832,0.432225


In [25]:
print("Before guardrails:", len(df_ranked))
print("After guardrails:", len(df_filtered))


Before guardrails: 10
After guardrails: 4


In [26]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Features for clustering (use normalized, strategy-relevant metrics)
cluster_features = [
    "economic_score",
    "growth_score",
    "readiness_score",
    "risk_score"
]

X = df_filtered[cluster_features]
X.head()


,economic_score,growth_score,readiness_score,risk_score
6,0.476904,0.376511,0.920061,0.543242
0,0.363644,0.400268,0.751150,0.477438
2,0.324748,0.389784,0.558764,0.451493
4,0.331591,0.000000,0.662832,0.432225


In [27]:
# Standardize features for clustering
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# KMeans clustering
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_filtered["cluster"] = kmeans.fit_predict(X_scaled)

df_filtered[["country", "cluster"]]


C:\Users\Jagat\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


,country,cluster
6,Singapore,2
0,Australia,0
2,Canada,0
4,Germany,1


In [31]:
# Cluster centroids for interpretation
cluster_summary = (
    df_filtered
    .groupby("cluster")[cluster_features]
    .mean()
    .round(3)
)

cluster_summary


,economic_score,growth_score,readiness_score,risk_score
cluster,,,,
0,0.344,0.395,0.655,0.464
1,0.332,0.000,0.663,0.432
2,0.477,0.377,0.920,0.543


In [32]:
def generate_insight(row):
    reasons = []
    
    # Strengths
    if row["readiness_score"] > 0.6:
        reasons.append("strong market readiness")
    if row["economic_score"] > 0.6:
        reasons.append("solid economic fundamentals")
    if row["growth_score"] > 0.6:
        reasons.append("high growth momentum")
    
    # Risks
    risks = []
    if row["risk_score"] < 0.5:
        risks.append("elevated operational or political risk")
    if row["cost_of_living_norm"] < 0.4:
        risks.append("high cost pressure")
    
    # Construct explanation
    insight = "Ranked high due to " + ", ".join(reasons) if reasons else "Moderate overall fundamentals"
    
    if risks:
        insight += "; however, risks include " + ", ".join(risks)
    
    return insight


In [33]:
df_filtered["ai_insight"] = df_filtered.apply(generate_insight, axis=1)

df_filtered[[
    "country",
    "filtered_rank",
    "final_score",
    "cluster",
    "ai_insight"
]]


,country,filtered_rank,final_score,cluster,ai_insight
6,Singapore,1,0.614694,2,Ranked high due to strong market readiness; ho...
0,Australia,2,0.519528,0,Ranked high due to strong market readiness; ho...
2,Canada,3,0.442732,0,"Moderate overall fundamentals; however, risks ..."
4,Germany,4,0.411415,1,Ranked high due to strong market readiness; ho...


In [34]:
cluster_labels = {
    0: "Stable & Execution-Ready",
    1: "High-Growth but Risk-Sensitive",
    2: "Large Market, Operationally Complex"
}

df_filtered["market_archetype"] = df_filtered["cluster"].map(cluster_labels)

df_filtered[["country", "market_archetype"]]


,country,market_archetype
6,Singapore,"Large Market, Operationally Complex"
0,Australia,Stable & Execution-Ready
2,Canada,Stable & Execution-Ready
4,Germany,High-Growth but Risk-Sensitive


In [35]:
df_ranked.columns


Index(['country', 'region', 'gdp', 'gdp_per_capita', 'gdp_growth',
       'population', 'youth_population_pct', 'internet_penetration',
       'mobile_broadband', 'tertiary_enrollment', 'outbound_mobility_ratio',
       'english_proficiency', 'ease_of_business', 'regulatory_quality',
       'political_stability', 'inflation', 'cost_of_living',
       'currency_stability', 'data_year', 'reliability_score', 'gdp_norm',
       'gdp_per_capita_norm', 'gdp_growth_norm', 'population_norm',
       'internet_penetration_norm', 'mobile_broadband_norm',
       'tertiary_enrollment_norm', 'english_proficiency_norm',
       'ease_of_business_norm', 'regulatory_quality_norm',
       'political_stability_norm', 'inflation_norm', 'cost_of_living_norm',
       'currency_stability_norm', 'economic_score', 'growth_score',
       'readiness_score', 'risk_score', 'final_score', 'rank'],
      dtype='object')

In [36]:
df_ranked.to_csv("../data/scoring_data.csv", index=False)
print("scoring_data.csv overwritten")


scoring_data.csv overwritten


In [37]:
# Save FINAL decision-ready dataset (includes archetypes + insights)
df_filtered.to_csv("../data/scoring_data.csv", index=False)
print("FINAL scoring_data.csv saved")


FINAL scoring_data.csv saved
